# Benchmarking GRN inference with BEELINE

This notebook shows off `iqcell.beeline`, which wraps the real [Murali-group/BEELINE](https://github.com/Murali-group/Beeline) pipeline. We start from a **known** ground-truth gene regulatory network, simulate scRNA-seq-like data with `iqcell.simulation`, export it in BEELINE's input layout, and score inference results.

Because BEELINE runs its algorithms in Docker containers, the notebook works in two modes:

- **No BEELINE/Docker** (default here): we export the inputs + config and use iqcell's pure-python `score_ranking` (AUPRC/AUROC) so everything runs end-to-end with no external setup.
- **With BEELINE**: point `BEELINE_REPO` at a cloned + initialized checkout (see [`docs/beeline.md`](../docs/beeline.md)) and the same cells will drive the real `BLRunner.py` / `BLEvaluator.py`.

## Setup

In [ ]:
import os, sys
# Make the repo root importable when running from examples/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from iqcell.simulation import GRNSpec, HillParams, SyntheticGRNGenerator, NoiseConfig
from iqcell.beeline import (
    export_beeline_inputs,
    write_config,
    BeelineRunner,
    read_ranked_edges,
    parse_evaluation,
    score_ranking,
)

# Point this at a bootstrapped BEELINE checkout to run the real pipeline,
# e.g. os.environ['BEELINE_REPO'] = '../.beeline' (see scripts/bootstrap_beeline.sh).
BEELINE_REPO = os.environ.get('BEELINE_REPO')
WORK_DIR = os.path.abspath('data/beeline_notebook')
INPUT_DIR = os.path.join(WORK_DIR, 'inputs')
OUTPUT_DIR = os.path.join(WORK_DIR, 'outputs')
CONFIG_PATH = os.path.join(WORK_DIR, 'config.yaml')
os.makedirs(INPUT_DIR, exist_ok=True)
print('work dir:', WORK_DIR)

## 1. Define the ground-truth GRN

A small hematopoiesis-inspired feed-forward network (kept acyclic). `Sox2` is the root driver; the rest are regulated. Edge signs are the *truth* we will later try to recover.

```
Sox2  -> Gata2
Gata2 -> Gata1,  Gata2 -> Pu1
Pu1   -| Gata1   (repression)
Gata1 -> Klf1,   Pu1 -| Klf1
```

In [ ]:
spec = GRNSpec(['Sox2', 'Gata2', 'Gata1', 'Pu1', 'Klf1'])
spec.set_rule('Gata2', activators=['Sox2'])
spec.set_rule('Gata1', activators=['Gata2'], repressors=['Pu1'])
spec.set_rule('Pu1', activators=['Gata2'])
spec.set_rule('Klf1', activators=['Gata1'], repressors=['Pu1'],
              hill={'Gata1': HillParams(K=0.4, n=6.0)})
spec.validate()

ground_truth = list(spec.edges())  # (regulator, target, sign)
ground_truth

Visualize the ground-truth network (green = activation, red = repression):

In [ ]:
G = nx.DiGraph()
G.add_nodes_from(spec.genes)
for reg, tgt, sign in ground_truth:
    G.add_edge(reg, tgt, sign=sign)

pos = nx.spring_layout(G, seed=1)
edge_colors = ['#2ca02c' if G[u][v]['sign'] == 1 else '#d62728' for u, v in G.edges()]
plt.figure(figsize=(5, 4))
nx.draw_networkx_nodes(G, pos, node_color='#dfe6ee', edgecolors='#33475b', node_size=1600)
nx.draw_networkx_labels(G, pos, font_size=10)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=2,
                       arrowsize=20, connectionstyle='arc3,rad=0.08')
plt.title('Ground-truth GRN'); plt.axis('off'); plt.tight_layout(); plt.show()

## 2. Simulate scRNA-seq-like data

`SyntheticGRNGenerator` produces continuous expression along pseudotime with biological noise and dropout, so the exported matrix looks like real assay data.

In [ ]:
gen = SyntheticGRNGenerator(
    spec, n_cells=500, reg_skip=8,
    noise=NoiseConfig(biological=0.05, dropout=0.2), seed=0,
).simulate()

expr = pd.DataFrame(gen.expression, columns=spec.genes,
                    index=[f'cell_{i}' for i in range(gen.n_cells)])
expr.head()

Expression along pseudotime for each gene:

In [ ]:
plt.figure(figsize=(7, 4))
for gene in spec.genes:
    plt.plot(gen.pseudotime, expr[gene], label=gene, alpha=0.8)
plt.xlabel('pseudotime'); plt.ylabel('expression'); plt.legend()
plt.title('Simulated expression trajectories'); plt.tight_layout(); plt.show()

## 3. Export BEELINE inputs + config

`export_beeline_inputs` writes `ExpressionData.csv` (genes x cells), `PseudoTime.csv`, and the ground-truth `refNetwork.csv`. `write_config` generates a BEELINE-compatible YAML enabling PIDC, GENIE3, and PEARSON.

In [ ]:
paths = export_beeline_inputs(gen, INPUT_DIR, dataset_id='synthetic', run_id='run1')
write_config(CONFIG_PATH, input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)
for key, path in paths.items():
    print(f'{key:11s} -> {os.path.relpath(path, WORK_DIR)}')

The ground-truth network as BEELINE sees it (`Gene1,Gene2,Type`):

In [ ]:
print(open(paths['network']).read())

## 4. Run inference (real BEELINE if available)

`BeelineRunner.check_available()` probes for the repo + Docker **without** invoking Docker. If a checkout is configured we run the real pipeline; otherwise we skip to the pure-python scorer below.

In [ ]:
ran_beeline = False
if BEELINE_REPO:
    runner = BeelineRunner(BEELINE_REPO)
    status = runner.check_available()
    print(status)
    if status.available:
        runner.run(CONFIG_PATH)
        runner.evaluate(CONFIG_PATH, auc=True, epr=True)
        print('BEELINE metrics:', parse_evaluation(OUTPUT_DIR))
        ran_beeline = True
else:
    print('BEELINE_REPO not set - using the pure-python scorer in step 5.')

## 5. Score rankings without BEELINE

`score_ranking` compares a ranked edge list to the known ground truth over all directed gene pairs (self-loops excluded), returning AUPRC/AUROC. It works with or without scikit-learn. Below we contrast an **oracle** ranking (true edges on top) against a **random** ranking to sanity-check the metric.

In [ ]:
all_pairs = [(a, b) for a in spec.genes for b in spec.genes if a != b]

oracle = pd.DataFrame(
    [(reg, tgt, 1.0) for reg, tgt, _ in ground_truth],
    columns=['Gene1', 'Gene2', 'EdgeWeight'],
)
rng = np.random.default_rng(0)
random_rank = pd.DataFrame(
    [(a, b, float(rng.random())) for a, b in all_pairs],
    columns=['Gene1', 'Gene2', 'EdgeWeight'],
)

print('oracle:', score_ranking(oracle, ground_truth))
print('random:', score_ranking(random_rank, ground_truth))

If a real BEELINE run produced `rankedEdges.csv` files, score each algorithm the same way:

In [ ]:
run_out = os.path.join(OUTPUT_DIR, 'synthetic', 'run1')
if os.path.isdir(run_out):
    for algo in sorted(os.listdir(run_out)):
        ranked = os.path.join(run_out, algo, 'rankedEdges.csv')
        if os.path.isfile(ranked):
            df = read_ranked_edges(ranked)
            print(algo, score_ranking(df, ground_truth))
else:
    print('No BEELINE outputs yet - run step 4 with a BEELINE_REPO set.')

## Next steps

- Bootstrap a real BEELINE checkout: `scripts/bootstrap_beeline.sh` (needs Docker).
- Set `os.environ['BEELINE_REPO'] = '../.beeline'` in the Setup cell and re-run steps 4-5 to compare real algorithms (PIDC, GENIE3, PEARSON, ...).
- See [`docs/beeline.md`](../docs/beeline.md) for the full workflow and troubleshooting.